# Lab 2.2 — Tune the vector index

**Before you start:** select **Cell > Run All** to initialize the harness.

Create `cortex-corpus-tuned` with `bbq_hnsw` index options that fit your seeded RAM budget, reindex, and verify that quantization does not cost meaningful relevance.

In [ ]:
# ── Harness setup (run once) ──────────────────────────────────────────────────
import sys, os, json, pathlib, time

sys.path.insert(0, '/opt/ara/lib')
from ara_metrics import retriever_hybrid_rrf, run_queries_with_template, hnsw_budget

env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())

from elasticsearch import Elasticsearch
ES_URL   = os.environ['ES_URL']
ES_KEY   = os.environ['ES_API_KEY']
EMBED_ID = os.environ.get('ARA_EMBED_ID', '.jina-embeddings-v5-text-small')

es = Elasticsearch(ES_URL, api_key=ES_KEY, request_timeout=120)

budget_info = json.loads(pathlib.Path('/home/elastic/dev-sets/03-ram-budget.json').read_text())
RAM_BUDGET_GB = budget_info['ram_budget_gb']
RAM_BUDGET_BYTES = int(RAM_BUDGET_GB * 1024**3)

dev_queries = json.loads(pathlib.Path('/home/elastic/dev-sets/dev-queries.json').read_text())

print(f'Harness ready. EMBED_ID={EMBED_ID}')
print(f'Production RAM budget: {RAM_BUDGET_GB:.1f} GB ({RAM_BUDGET_BYTES:,} bytes)')

---
## Step 1 — Find the largest m that fits your budget

Budget formula: `bytes = (128 + m × 4) × 12_000_000`
- `128` bytes = BBQ compressed vector (1 bit/dim × 1024 dims = 128 bytes)
- `m × 4` bytes = HNSW neighbor pointers

Use `hnsw_budget(n_vectors, dims, m, quantization)` from `ara_metrics`.

In [ ]:
# ── YOUR WORK ── Find the largest m within your RAM budget ───────────────────
N_VECTORS = 12_000_000
DIMS = 1024

# Try values of m and find the largest that fits
chosen_m = None
for m in range(4, 65):
    estimated = hnsw_budget(N_VECTORS, DIMS, m, 'bbq')
    if estimated <= RAM_BUDGET_BYTES:
        chosen_m = m
    else:
        break

print(f'Largest m that fits {RAM_BUDGET_GB:.1f} GB: m={chosen_m}')
print(f'Estimated budget: {hnsw_budget(N_VECTORS, DIMS, chosen_m, "bbq") / 1024**3:.2f} GB')

---
## Step 2 — Create cortex-corpus-tuned

In **Kibana Dev Tools**, create the index with your chosen `m` and an `ef_construction` of at least 100:

```json
PUT /cortex-corpus-tuned
{
  "settings": {"number_of_shards": 1},
  "mappings": {
    "properties": {
      "body": {
        "type": "semantic_text",
        "inference_id": "<ARA_EMBED_ID>",
        "index_options": {
          "type": "bbq_hnsw",
          "m": <your m>,
          "ef_construction": <at least 100>
        }
      },
      "body_text": {"type": "text"},
      "doc_id": {"type": "keyword"},
      "doc_type": {"type": "keyword"},
      "title": {"type": "keyword"},
      "section": {"type": "keyword"},
      "effective_date": {"type": "date"}
    }
  }
}
```

Then reindex:
```json
POST /cortex-corpus/_reindex
{"dest": {"index": "cortex-corpus-tuned"}}
```

In [ ]:
# ── Measure nDCG delta after reindexing ──────────────────────────────────────
# Run after the reindex completes (check doc count first)
from ara_metrics import retriever_hybrid_rrf, run_queries_with_template, ndcg_at_k

hybrid_tmpl = retriever_hybrid_rrf(text_field='body', semantic_field='body')

untuned = run_queries_with_template(es, 'cortex-corpus',       dev_queries, hybrid_tmpl, k=5, n_passes=3)
tuned   = run_queries_with_template(es, 'cortex-corpus-tuned', dev_queries, hybrid_tmpl, k=5, n_passes=3)

ndcg_untuned = ndcg_at_k(untuned, dev_queries, k=5)
ndcg_tuned   = ndcg_at_k(tuned,   dev_queries, k=5)
delta        = abs(ndcg_tuned - ndcg_untuned)

print(f'Untuned  hybrid nDCG@5: {ndcg_untuned:.3f}')
print(f'Tuned    hybrid nDCG@5: {ndcg_tuned:.3f}')
print(f'Delta:                  {delta:.3f}  (noise band: 0.03)')
print(f'Result: {"WITHIN noise band" if delta <= 0.03 else "EXCEEDS noise band — quantization cost relevance"}')

In [ ]:
# ── Save tuning results ───────────────────────────────────────────────────────
results = {
    'ndcg_untuned': ndcg_untuned,
    'ndcg_tuned': ndcg_tuned,
    'ndcg_delta': delta,
    'chosen_m': chosen_m,
}
pathlib.Path('/home/elastic/tuning-results.json').write_text(json.dumps(results, indent=2))
print('tuning-results.json saved. Select Check in the sidebar.')